# In-Class Assignment — Python for Feature Extraction
**Time:** 20 minutes  |  **Points:** 20


....


Dataset file: `product_reviews.txt`

## Load the dataset
Download it from **Canvas**, then run the upload cell below to select the file from your computer.


In [1]:
# Upload the dataset file from your computer (Google Colab)
from google.colab import files

uploaded = files.upload()   # choose product_reviews.txt
filename = next(iter(uploaded))
print("Uploaded file name:", filename)


Saving product_reviews.txt to product_reviews.txt
Uploaded file name: product_reviews.txt


## Q1 (2 point) — Load & Read & inspect

## Note about the header
**Important:** The dataset file includes a **header row** (column names).  
Make sure the header is used as the column names — it **should NOT appear as a data row** in your DataFrame.

Tip: Use the filename variable printed above when reading the file.
After Reading, check that your columns are `id` and `text`.
Print: **(a)** `df.shape`, **(b)** `df.head(3\)`.  


In [15]:
import pandas as pd

# Read file WITH header (default behavior)
df = pd.read_csv(filename, sep='\t')

# Verify columns
print("Columns:", df.columns)

# Required outputs
print("Shape:", df.shape)
df.head(3)



Columns: Index(['id', 'text'], dtype='object')
Shape: (10, 2)


,id,text
0,1,Love this blender! Smoothies are super creamy ...
1,2,Terrible quality... stopped working after 2 da...
2,3,Good value for the price. Shipping was quick.


## Q2 (4 points) — Basic handcrafted features  
Create these columns and then display the DataFrame:
- `word_count` = number of words  
- `char_count` = number of characters  
- `avg_word_len` = average word length (ignore punctuation)  
- `excl_count` = number of `!` characters  

Print: `id, word_count, char_count, avg_word_len, excl_count`.


In [16]:
import re

# Word count
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

# Character count
df['char_count'] = df['text'].apply(lambda x: len(str(x)))

# Average word length (ignore punctuation)
def avg_word_len(text):
    words = re.findall(r'\b\w+\b', str(text))  # keeps only words, removes punctuation
    if len(words) == 0:
        return 0
    return sum(len(word) for word in words) / len(words)

df['avg_word_len'] = df['text'].apply(avg_word_len)

# Exclamation count
df['excl_count'] = df['text'].apply(lambda x: str(x).count('!'))

# Print required columns
df[['id', 'word_count', 'char_count', 'avg_word_len', 'excl_count']]



,id,word_count,char_count,avg_word_len,excl_count
0,1,9,55,5.000000,1
1,2,7,51,5.571429,3
2,3,8,45,4.500000,0
3,4,10,56,4.500000,0
4,5,8,53,5.500000,1
5,6,10,51,4.000000,0
6,7,9,50,4.444444,0
7,8,6,40,5.500000,0
8,9,7,52,6.285714,0
9,10,7,48,4.750000,0


## Q3 (6 points) — Bag-of-Words (CountVectorizer)  
Use `CountVectorizer(stop_words="english")` on `df["text"]`. Print:
1) vocabulary size (number of features)  
2) top 10 words by **total count** across all documents (word + count)


In [18]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["text"])

vocab_size = len(vectorizer.get_feature_names_out())
print("Vocabulary size:", vocab_size)



Vocabulary size: 50


In [19]:
import numpy as np

# Sum counts of each word
word_counts = np.array(X.sum(axis=0)).flatten()

# Get feature names (words)
words = vectorizer.get_feature_names_out()

# Combine words + counts
word_freq = list(zip(words, word_counts))

# Sort by count descending
word_freq_sorted = sorted(word_freq, key=lambda x: x[1], reverse=True)

# Top 10
top_10 = word_freq_sorted[:10]

print("Top 10 words by total count:")
for word, count in top_10:
    print(word, count)

Top 10 words by total count:
amazing 1
app 1
battery 1
blender 1
box 1
buy 1
charged 1
clear 1
crashing 1
creamy 1


## Q4 (4 points) — Bigram features  
Use `CountVectorizer(stop_words="english", ngram_range=(2,2))`.  
Print the top 5 bigrams by total count (bigram + count).


In [20]:
from sklearn.feature_extraction.text import CountVectorizer

bigram_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(2, 2)   # ONLY bigrams
)

X_bigram = bigram_vectorizer.fit_transform(df["text"])

import numpy as np

bigram_counts = np.array(X_bigram.sum(axis=0)).flatten()
bigrams = bigram_vectorizer.get_feature_names_out()

bigram_freq = list(zip(bigrams, bigram_counts))

# Sort descending
bigram_freq_sorted = sorted(bigram_freq, key=lambda x: x[1], reverse=True)

top_5_bigrams = bigram_freq_sorted[:5]

print("Top 5 bigrams by total count:")
for bg, count in top_5_bigrams:
    print(bg, count)


Top 5 bigrams by total count:
amazing screen 1
app keeps 1
battery life 1
blender smoothies 1
box damaged 1


## Q5 (4 points) — TF-IDF features  
Use `TfidfVectorizer(stop_words="english", ngram_range=(1,2))`.  
Compute the **average TF-IDF** score of each term across documents and print the top 5 terms (term + avg score).


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(df["text"])

avg_tfidf = np.array(X_tfidf.mean(axis=0)).flatten()
terms = tfidf.get_feature_names_out()

top_5 = sorted(zip(terms, avg_tfidf), key=lambda x: x[1], reverse=True)[:5]

for term, score in top_5:
    print(term, score)



clear 0.03779644730092272
easy 0.03779644730092272
easy instructions 0.03779644730092272
instructions 0.03779644730092272
instructions clear 0.03779644730092272


## Grading Checklist
- Q1: correct prints  
- Q2: correct feature columns + requested display  
- Q3: correct vocabulary size + correct top 10 words by total count  
- Q4: correct top 5 bigrams by total count  
- Q5: correct top 5 TF-IDF terms by average score
